# Photo Renamer

Renames a set of images by their EXIF capture date. Images are renamed to `<name>_<number>.<primary_ext>`, ordered chronologically. Secondary-format images (e.g. JPEGs created alongside RAW files with the same stem) are renamed to match.

# Dependencies

In [1]:
import os
from datetime import datetime
from ledsa.core.image_reading import get_exif_entry

# Configuration

In [2]:
# Directory containing the images to rename
working_dir = "/path/to/your/images"

# Base name used for renamed files
name = "IMG"

# Primary file type (e.g. CR3, CR2, RAW) — these are sorted by EXIF capture date
primary_ext = "CR3"

# Secondary file type sharing the same stem as each primary file (e.g. JPG) — set to None to skip
secondary_ext = "JPG"

# Zero-padding width for the numeric counter (e.g. 4 → IMG_0001.CR3)
padding = 4

# Scan and Sort Images

In [3]:
primary_ext_norm = primary_ext.lstrip('.')
secondary_ext_norm = secondary_ext.lstrip('.') if secondary_ext else None

# Collect all primary files
primary_files = [
    f for f in os.listdir(working_dir)
    if f.upper().endswith('.' + primary_ext_norm.upper())
]

if not primary_files:
    raise FileNotFoundError(f"No .{primary_ext_norm} files found in {working_dir}")

print(f"Found {len(primary_files)} .{primary_ext_norm} file(s). Reading EXIF capture dates...\n")

# Read capture date for each primary file
records = []
failed = []
for fname in primary_files:
    fpath = os.path.join(working_dir, fname)
    try:
        raw_date = get_exif_entry(fpath, 'DateTimeOriginal')
        capture_dt = datetime.strptime(raw_date.strip(), "%Y:%m:%d %H:%M:%S")
        records.append((fname, capture_dt))
    except Exception as e:
        failed.append((fname, str(e)))

if failed:
    print("WARNING: Could not read capture date for the following files (they will be skipped):")
    for fname, err in failed:
        print(f"  {fname}: {err}")
    print()

# Sort by capture date
records.sort(key=lambda x: x[1])

# Build rename plan
rename_plan = []  # list of (old_path, new_path, old_secondary_path, new_secondary_path)

for idx, (fname, capture_dt) in enumerate(records, start=1):
    stem = os.path.splitext(fname)[0]
    new_stem = f"{name}_{idx:0{padding}d}"

    old_primary = os.path.join(working_dir, fname)
    new_primary = os.path.join(working_dir, f"{new_stem}.{primary_ext_norm}")

    old_secondary = None
    new_secondary = None
    if secondary_ext_norm:
        # Try both the same case and upper/lower variants
        for sec_candidate in [f"{stem}.{secondary_ext_norm}",
                               f"{stem}.{secondary_ext_norm.upper()}",
                               f"{stem}.{secondary_ext_norm.lower()}"]:
            candidate_path = os.path.join(working_dir, sec_candidate)
            if os.path.exists(candidate_path):
                ext_used = os.path.splitext(sec_candidate)[1].lstrip('.')
                old_secondary = candidate_path
                new_secondary = os.path.join(working_dir, f"{new_stem}.{ext_used}")
                break

    rename_plan.append((old_primary, new_primary, old_secondary, new_secondary, capture_dt))

# Print renaming report
col_w = max(len(os.path.basename(r[0])) for r in rename_plan)
print(f"{'Original':<{col_w}}  {'Capture date':<22}  {'New name'}")
print("-" * (col_w + 22 + 20))
for old_p, new_p, old_s, new_s, dt in rename_plan:
    old_name = os.path.basename(old_p)
    new_name = os.path.basename(new_p)
    date_str = dt.strftime("%Y-%m-%d %H:%M:%S")
    sec_info = f"  +  {os.path.basename(old_s)} → {os.path.basename(new_s)}" if old_s else ""
    print(f"{old_name:<{col_w}}  {date_str:<22}  {new_name}{sec_info}")

print(f"\nTotal: {len(rename_plan)} primary file(s) to rename" +
      (f", {sum(1 for r in rename_plan if r[2])} with a matching {secondary_ext_norm} file" if secondary_ext_norm else "") + ".")

Found 667 .CR2 file(s). Reading EXIF capture dates...

Original      Capture date            New name
------------------------------------------------------
IMG_6493.CR2  2026-05-04 13:53:16     V001_0001.CR2  +  IMG_6493.JPG → V001_0001.JPG
IMG_6494.CR2  2026-05-04 13:53:35     V001_0002.CR2  +  IMG_6494.JPG → V001_0002.JPG
IMG_6495.CR2  2026-05-04 13:53:50     V001_0003.CR2  +  IMG_6495.JPG → V001_0003.JPG
IMG_6496.CR2  2026-05-04 13:53:58     V001_0004.CR2  +  IMG_6496.JPG → V001_0004.JPG
IMG_6497.CR2  2026-05-04 13:54:18     V001_0005.CR2  +  IMG_6497.JPG → V001_0005.JPG
IMG_6498.CR2  2026-05-04 13:54:25     V001_0006.CR2  +  IMG_6498.JPG → V001_0006.JPG
IMG_6499.CR2  2026-05-04 13:55:22     V001_0007.CR2  +  IMG_6499.JPG → V001_0007.JPG
IMG_6500.CR2  2026-05-04 13:55:33     V001_0008.CR2  +  IMG_6500.JPG → V001_0008.JPG
IMG_6501.CR2  2026-05-04 13:57:08     V001_0009.CR2  +  IMG_6501.JPG → V001_0009.JPG
IMG_6502.CR2  2026-05-04 13:57:14     V001_0010.CR2  +  IMG_6502.JPG → V001_00

# Rename Images

Run this cell to apply the renaming shown above. You will be asked to confirm before any files are changed.

In [4]:
answer = input("Proceed with renaming? Type 'yes' to confirm: ").strip().lower()

if answer != 'yes':
    print("Renaming cancelled.")
else:
    renamed_primary = 0
    renamed_secondary = 0
    errors = []

    for old_p, new_p, old_s, new_s, _ in rename_plan:
        try:
            if old_p != new_p:
                os.rename(old_p, new_p)
                renamed_primary += 1
        except Exception as e:
            errors.append(f"  {os.path.basename(old_p)} → {os.path.basename(new_p)}: {e}")

        if old_s and new_s:
            try:
                if old_s != new_s:
                    os.rename(old_s, new_s)
                    renamed_secondary += 1
            except Exception as e:
                errors.append(f"  {os.path.basename(old_s)} → {os.path.basename(new_s)}: {e}")

    print(f"Done. Renamed {renamed_primary} primary file(s)" +
          (f" and {renamed_secondary} secondary file(s)" if secondary_ext_norm else "") + ".")

    if errors:
        print("\nErrors encountered:")
        for err in errors:
            print(err)

Done. Renamed 667 primary file(s) and 667 secondary file(s).
